In [1]:
# Se importan las librerías usadas para construir el mart analítico.

from pathlib import Path
import sqlite3

import pandas as pd
import plotly.express as px

In [2]:
# Se definen las rutas de fuentes y del mart permanente.

data_directory = Path("../data")
submission_directory = Path("../submission")
submission_directory.mkdir(exist_ok=True)
mart_file = submission_directory / "sales_mart.db"

In [3]:
# Se cargan las fuentes operativas que alimentan el hecho de ventas.

order_lines = pd.read_csv(data_directory / "order_lines.csv")
customers = pd.read_csv(data_directory / "customers.csv")
products = pd.read_csv(data_directory / "products.csv")
order_lines.shape, customers.shape, products.shape

((471, 7), (60, 3), (12, 4))

In [4]:
# Se define el hecho con grano línea de pedido antes de crear dimensiones.

facts = order_lines.merge(
    products[["product_id", "unit_price"]], on="product_id", validate="many_to_one"
)
facts["order_date"] = pd.to_datetime(facts["order_date"])
facts["gross_sales"] = facts["quantity"] * facts["unit_price"]
facts["net_sales"] = facts["gross_sales"] * (1 - facts["discount_pct"])
facts[["order_id", "line_id"]].duplicated().sum(), facts.shape

(0, (471, 10))

In [5]:
# Se construyen dimensiones para fecha, cliente y producto.

dim_date = pd.DataFrame({"date": sorted(facts["order_date"].unique())})
dim_date["date_key"] = range(1, len(dim_date) + 1)
dim_date["year"] = dim_date["date"].dt.year
dim_date["month"] = dim_date["date"].dt.month
dim_customer = customers.assign(customer_key=customers["customer_id"])
dim_product = products.assign(product_key=products["product_id"])
dim_date.shape, dim_customer.shape, dim_product.shape

((240, 4), (60, 4), (12, 5))

In [6]:
# Se reemplazan identificadores operativos por claves del mart sin cambiar el número de hechos.

fact_sales = facts.merge(
    dim_date[["date", "date_key"]],
    left_on="order_date",
    right_on="date",
    validate="many_to_one",
)[
    [
        "order_id",
        "line_id",
        "date_key",
        "customer_id",
        "product_id",
        "quantity",
        "gross_sales",
        "discount_pct",
        "net_sales",
    ]
].rename(
    columns={"customer_id": "customer_key", "product_id": "product_key"}
)
assert len(fact_sales) == len(facts)
assert fact_sales[["date_key", "customer_key", "product_key"]].notna().all().all()
fact_sales.head()

,order_id,line_id,date_key,customer_key,product_key,quantity,gross_sales,discount_pct,net_sales
0,1,1,2,51,12,5,2450,0.05,2327.5
1,1,2,2,51,1,4,3120,0.10,2808.0
2,1,3,2,51,3,2,1020,0.00,1020.0
3,2,1,4,35,7,5,1200,0.05,1140.0
4,3,1,6,16,6,5,1450,0.10,1305.0


In [7]:
# Se publica el mart estrella para consultas analíticas posteriores.

with sqlite3.connect(mart_file) as connection:
    dim_date.to_sql("dim_date", connection, index=False, if_exists="replace")
    dim_customer.to_sql("dim_customer", connection, index=False, if_exists="replace")
    dim_product.to_sql("dim_product", connection, index=False, if_exists="replace")
    fact_sales.to_sql("fact_sales", connection, index=False, if_exists="replace")

In [8]:
# ¿Qué combinación de región y categoría aporta más ventas netas en el mart?

query = """
SELECT c.region, p.category, SUM(f.net_sales) AS net_sales
FROM fact_sales f
JOIN dim_customer c USING(customer_key)
JOIN dim_product p USING(product_key)
GROUP BY c.region, p.category
ORDER BY net_sales DESC
"""
with sqlite3.connect(mart_file) as connection:
    region_category = pd.read_sql_query(query, connection)
region_category

,region,category,net_sales
0,Centro,Servicios,106963.0
1,Sur,Tecnología,99607.5
2,Norte,Servicios,92853.0
3,Sur,Servicios,91112.5
4,Centro,Tecnología,85582.0
5,Norte,Tecnología,77188.5
6,Norte,Oficina,44838.5
7,Centro,Oficina,39462.5
8,Sur,Oficina,35383.0


In [9]:
# Se comparan regiones y categorías mediante una visualización de medidas agregadas.

fig = px.bar(
    region_category,
    x="region",
    y="net_sales",
    color="category",
    barmode="group",
    title="Ventas netas por región y categoría",
    labels={"region": "Región", "net_sales": "Ventas netas", "category": "Categoría"},
)
fig.update_layout(template="plotly_white")
fig.show()